# Labeled Data

In [ ]:
import polars as pl
def flatten_journeys_parquet(input_csv_path, output_parquet_path):
    q = pl.scan_csv(input_csv_path)

    q = q.unique(subset=["id", "ed_id", "event_timestamp"])

    df = (
        q
        .with_columns(
            pl.col("event_timestamp").str.to_datetime(time_zone="UTC")
        )
        .group_by("id")
        .agg(
            # 1. Create the nested journey list
            pl.struct(["event_timestamp", "ed_id"])
            .sort_by(["event_timestamp", "ed_id"])
            .alias("journey"),
            
            # 2. Mark as successful if ed_id 28 exists anywhere in the journey
            (pl.col("ed_id") == 28).any().alias("is_success")
        )
    )

    df.sink_parquet(output_parquet_path)
    
    
training_csv_path = "/Users/emiliodulay/Documents/1. UCLA/2. Year 2/3. Spring 2026/STAT M148/statM148proj/train.csv"
output_path = "/Users/emiliodulay/Documents/1. UCLA/2. Year 2/3. Spring 2026/STAT M148/statM148proj/train_labeled_flattened.parquet"

flatten_journeys_parquet(training_csv_path, output_path)

In [ ]:
import torch
import torch.nn as nn
import polars as pl
from time_aware_transformer import make_classification_transformer
from time_dataset import prepare_datasets

In [2]:
df = pl.scan_parquet("/Users/emiliodulay/Documents/1. UCLA/2. Year 2/3. Spring 2026/STAT M148/statM148proj/training_labeled_flattened.parquet")
df.head().collect()

id,journey,is_success
str,list[struct[2]],bool
"""-635794259 449271353""","[{2021-10-19 06:00:00 UTC,2}, {2021-10-19 22:01:11 UTC,12}, … {2022-06-05 10:36:29 UTC,1}]",false
"""-286026616 -1216078057""","[{2021-11-04 08:16:13 UTC,4}, {2021-11-04 08:16:18 UTC,11}, … {2022-12-13 18:08:26 UTC,4}]",false
"""-71256168 -2101645951""","[{2022-06-06 06:00:00 UTC,2}, {2022-06-06 10:38:46 UTC,12}, … {2022-07-11 00:00:00 UTC,28}]",true
"""-251999408 -860325245""","[{2022-05-26 06:00:00 UTC,2}, {2022-05-26 19:44:28 UTC,19}, … {2022-06-28 10:10:30 UTC,4}]",false
"""1442883013 -1021362550""","[{2021-02-13 06:00:00 UTC,2}, {2021-02-13 13:08:56 UTC,12}, … {2021-02-16 00:00:00 UTC,28}]",true


# Build Label dict and split ID's

In [3]:
batch = df.select(["id", "is_success"]).collect()
label_dict = dict(zip(batch["id"], batch["is_success"]))

ids = batch["id"].to_list()
split = int(len(ids) * 0.8)
train_ids, val_ids = set(ids[:split]), set(ids[split:])

train_df = df.filter(pl.col("id").is_in(train_ids)).select(["id", "journey"])
val_df   = df.filter(pl.col("id").is_in(val_ids)).select(["id", "journey"])

train_labels = {k: v for k, v in label_dict.items() if k in train_ids}
val_labels   = {k: v for k, v in label_dict.items() if k in val_ids}

# Build DataLoaders

In [ ]:
train_loader, val_loader, _, scaler = prepare_datasets(
    train_df, val_df, val_df,
    train_labels, val_labels, val_labels,
    max_seq_len=256,
    batch_size=32,
)

# Insepect Data

In [ ]:
from time_dataset import inspect
inspect(train_df.collect(), n=3)

# Build model, optimizer, loss

In [ ]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = make_classification_transformer(
    num_features=7,   # must match len(FEATURE_COLS) in dataset.py
    num_classes=2,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

# Train

In [ ]:
for epoch in range(20):
    # ── Training ──
    model.train()
    total_loss = 0
    for batch in train_loader:
        src   = batch["src"].to(device)
        ts    = batch["timestamps"].to(device)
        mask  = batch["padding_mask"].to(device)
        label = batch["label"].to(device)

        logits = model(src, timestamps=ts, src_padding_mask=mask)
        loss   = criterion(logits, label)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ── Validation ──
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                batch["src"].to(device),
                timestamps=batch["timestamps"].to(device),
                src_padding_mask=batch["padding_mask"].to(device),
            )
            preds    = logits.argmax(dim=-1)
            correct += (preds == batch["label"].to(device)).sum().item()
            total   += len(preds)

    print(f"Epoch {epoch+1:02d}  loss={avg_loss:.4f}  val_acc={correct/total:.3f}")

# Performance

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# ── Collect all val predictions ──
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in val_loader:
        logits = model(
            batch["src"].to(device),
            timestamps=batch["timestamps"].to(device),
            src_padding_mask=batch["padding_mask"].to(device),
        )
        probs  = torch.softmax(logits, dim=-1)
        preds  = logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["label"].tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())   # prob of class 1 (success)

# ── 1. Classification report ──
print(classification_report(
    all_labels, all_preds,
    target_names=["not successful", "successful"]
))

# ── 2. ROC-AUC ──
auc = roc_auc_score(all_labels, all_probs)
print(f"ROC-AUC: {auc:.4f}")

# ── 3. Confusion matrix ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(all_labels, all_preds)
ConfusionMatrixDisplay(cm, display_labels=["not successful", "successful"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("Confusion Matrix")

# ── 4. ROC curve ──
RocCurveDisplay.from_predictions(
    all_labels, all_probs,
    name=f"Transformer (AUC={auc:.3f})",
    ax=axes[1]
)
axes[1].plot([0, 1], [0, 1], "k--", label="Random")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.tight_layout()
plt.savefig("model_performance.png", dpi=150)
plt.show()

# Save Model

In [ ]:
torch.save(model.state_dict(), "time_aware_transformer.pt")
print("Model saved.")

In [ ]:
model = make_classification_transformer(num_features=7, num_classes=2).to(device)
model.load_state_dict(torch.load("time_aware_transformer.pt", map_location=device))

# Test

In [ ]:
test_df = pl.scan_parquet("/Users/emiliodulay/Documents/1. UCLA/2. Year 2/3. Spring 2026/STAT M148/statM148proj/testing_new_data.parquet")

test_collected = test_df.select(["id", "journey"]).collect()
dummy_labels   = {id_: 0 for id_ in test_collected["id"].to_list()}

_, _, test_loader, _ = prepare_datasets(
    train_df, val_df, test_df.select(["id", "journey"]),
    train_labels, val_labels, dummy_labels,
    max_seq_len=256,
    batch_size=32,
)

model.eval()
all_ids, all_preds, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        logits = model(
            batch["src"].to(device),
            timestamps=batch["timestamps"].to(device),
            src_padding_mask=batch["padding_mask"].to(device),
        )
        probs = torch.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)

        all_ids.extend(batch["id"])
        all_preds.extend(preds.cpu().tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

submission = pl.DataFrame({
    "id":          all_ids,
    "predicted":   all_preds,
    "prob_success": all_probs,
})

submission.write_csv("submission.csv")
print(submission.head()) 